In [155]:
import random as r

def roulette(american=False, on_black=True):

    if american:
        n = r.randint(0,36)
    else:
        n = r.randint(0,37)

    if on_black:
        modulo = 1
    else:
        modulo = 0
    
    if n == 0 or n == 37 or n%2 != modulo:
        return False
    else:
        return True

def backup_plan(start_fund=10000, chosen_bet=0, buy_out=0, updates=True):

    funds = start_fund

    if chosen_bet <= 0:
        chosen_bet = funds/100

    if buy_out <= funds:
        buy_out = funds * 1.25

    tries, wins, loss_streak, max_loss_streak, max_bet, close_calls = 0, 0, 0, 0, 0, 0
    get_out = False
    current_bet = chosen_bet

    while current_bet < funds and funds < buy_out and not get_out:

        if roulette():
            funds += current_bet

            if current_bet > max_bet:
                max_bet = current_bet

            if loss_streak > max_loss_streak:
                max_loss_streak = loss_streak
            
            if current_bet*2 >= funds-current_bet:
                close_calls += 1

            current_bet = chosen_bet
            wins += 1
            loss_streak = 0

        else:
            funds -= current_bet
            current_bet *= 2
            loss_streak += 1

            if funds - current_bet < start_fund * .5:
                get_out = True
        
        tries += 1
    
    if current_bet >= funds:
        if updates:
            print(f"game ended due to not enough funds. \n {tries} plays \n ${funds} funds \n ${current_bet} max bet \n {close_calls} close calls")
        return tries, wins, max_loss_streak, funds, current_bet, close_calls, 0
    
    elif funds >= buy_out:
        if updates:
            print(f"game ended due to reaching buy out. \n {tries} plays \n ${funds} funds \n ${max_bet} max bet \n {close_calls} close calls")
        return tries, wins, max_loss_streak, funds, max_bet, close_calls, 1
    
    elif get_out:
        if updates:
            print(f"game ended due to passing get out threshold. \n {tries} plays \n ${funds} funds \n ${max_bet} max bet \n {close_calls} close calls")
        return tries, wins, max_loss_streak, funds, current_bet, close_calls, 2

# Single Play

In [156]:
funds = 10000

bet = funds/100

backup_plan(funds, bet)

game ended due to passing get out threshold. 
 28 plays 
 $8100.0 funds 
 $1600.0 max bet 
 0 close calls


(28, 12, 4, 8100.0, 3200.0, 0, 2)

# Monte Carlo

In [ ]:
def MonteCarloRoulette(iterations=100, funds=10000):
    """
    Reason Index:
    0: not enough funds
    1: reached buy out
    2: get out threshold triggered
    """
    exit_reason = [0,0,0]

    iter = 0
    winnings = 0
    tot_tries = 0
    tot_wins = 0
    tot_loss_streaks = 0

    reason_order = []
    
    while iter<iterations:
        iter += 1

        tries, wins, loss_streak, rake, max_bet, close_calls, reason_index = backup_plan(funds, chosen_bet=100, buy_out=funds*1.25, updates=False)
        exit_reason[reason_index] += 1
        if reason_index == 0:
            reason_order.append("Fund Out")
        elif reason_index == 1:
            reason_order.append("Buy Out")
        else:
            reason_order.append("Get Out")
        
        winnings += funds-rake
        tot_tries += tries
        tot_wins += wins
        tot_loss_streaks += loss_streak
    
    print("****Stats****")
    print("        Average Tries:", tot_tries/iterations)
    print("        Average Wins:", tot_wins/iterations)
    print("        Average Loss Streak:", tot_loss_streaks/iterations)
    print("        Winnings:", winnings)
    print("        First Ten Rounds:", reason_order[:10])

MonteCarloRoulette()

****Stats****
        Average Tries: 31.94
        Average Wins: 14.86
        Average Loss Streak: 2.91
        Winnings: 72800
        First Ten Rounds: ['Buy Out', 'Buy Out', 'Fund Out', 'Buy Out', 'Get Out', 'Get Out', 'Get Out', 'Buy Out', 'Get Out', 'Get Out']


# Monte Carlo of Monte Carlos

In [176]:
def run_simulations(num_simulations=500, iterations_per_sim=100, funds=1000000):
    """
    Each simulation = iterations_per_sim independent games of backup_plan().
    Tracks cumulative net P&L within each simulation to find peaks and valleys.

    Exit reason index:
      0: ran out of funds
      1: reached buy-out target (win)
      2: triggered get-out threshold
    """
    sim_nets = []
    sim_mins = []
    sim_maxs = []
    sim_winrates = []

    for _ in range(num_simulations):
        cumulative = 0
        running_min = 0
        running_max = 0
        bought_out = 0

        for _ in range(iterations_per_sim):
            tries, wins, loss_streak, rake, max_bet, close_calls, reason = backup_plan(
                funds, chosen_bet=10, buy_out=funds * 1.001, updates=False
            )
            net = rake - funds
            cumulative += net
            running_min = min(running_min, cumulative)
            running_max = max(running_max, cumulative)
            if reason == 1:
                bought_out += 1

        sim_nets.append(cumulative)
        sim_mins.append(running_min)
        sim_maxs.append(running_max)
        sim_winrates.append(bought_out / iterations_per_sim)

    n = num_simulations
    print(f"=== {n} simulations x {iterations_per_sim} games each ===\n")
    print(f"Avg net per simulation:       ${sum(sim_nets)/n:>10,.2f}")
    print(f"Best simulation:              ${max(sim_nets):>10,.2f}")
    print(f"Worst simulation:             ${min(sim_nets):>10,.2f}")
    print(f"\nAvg lowest point reached:   ${sum(sim_mins)/n:>10,.2f}")
    print(f"Worst low across all sims:    ${min(sim_mins):>10,.2f}")
    print(f"\nAvg highest point reached:  ${sum(sim_maxs)/n:>10,.2f}")
    print(f"Best high across all sims:    ${max(sim_maxs):>10,.2f}")
    print(f"\nAvg buy-out rate:           {sum(sim_winrates)/n*100:>9.1f}%")
    print(f"Avg net per game:             ${sum(sim_nets)/n/iterations_per_sim:>10,.2f}")

run_simulations()

=== 500 simulations x 100 games each ===

Avg net per simulation:       $-108,696.80
Best simulation:              $100,000.00
Worst simulation:             $-1,540,260.00

Avg lowest point reached:   $-181,524.80
Worst low across all sims:    $-1,563,260.00

Avg highest point reached:  $ 73,862.00
Best high across all sims:    $100,000.00

Avg buy-out rate:                99.4%
Avg net per game:             $ -1,086.97
